# Step 8 - Model Identification

This notebook translates the earlier diagnostics into an explicit modeling strategy. Instead of jumping straight to a large search grid, the goal is to choose the model family, evaluation window, and differencing structure that make sense for this hourly demand series.


In [1]:
import os

import pandas as pd
from IPython.display import display

PROCESSED_PATH = "../data/processed/"

transformation_selection = pd.read_csv(
    os.path.join(PROCESSED_PATH, "transformation_selection.csv")
)
seasonality_summary = pd.read_csv(os.path.join(PROCESSED_PATH, "seasonality_summary.csv"))
acf_pacf_guidance = pd.read_csv(os.path.join(PROCESSED_PATH, "acf_pacf_guidance.csv"))

display(transformation_selection)
display(seasonality_summary)
display(acf_pacf_guidance)


,selected_transformation,reason,seasonal_period_for_modeling
0,log_diff_1_diff_24,Balances variance stabilization with non-seaso...,24


,recommended_primary_seasonal_period,secondary_cycle_observed_hours,hourly_profile_range_mw,weekday_profile_range_mw,monthly_profile_range_mw,raw_autocorr_lag_24,raw_autocorr_lag_168
0,24,168,11011.705183,3865.676464,10020.47476,0.89141,0.782457


,component,recommended_values,reason
0,non_seasonal_ar_order_p,1 or 2,PACF typically shows its clearest spikes at th...
1,non_seasonal_ma_order_q,0 or 1,ACF usually drops quickly after the first few ...
2,seasonal_ar_order_P,0 or 1,Seasonal structure remains visible but not dom...
3,seasonal_ma_order_Q,0 or 1,Seasonal spikes around lag 24 often align with...
4,seasonal_period_s,24,The strongest repeating cycle in the raw serie...


In [2]:
model_family_comparison = pd.DataFrame(
    [
        {
            "model_family": "ARIMA",
            "use_case": "Non-seasonal baseline on differenced data",
            "fit_for_this_project": "Weak",
            "reason": "Ignores the dominant daily cycle and will likely leave seasonal autocorrelation in residuals.",
        },
        {
            "model_family": "SARIMA with s=24",
            "use_case": "Primary hourly forecasting candidate",
            "fit_for_this_project": "Strong",
            "reason": "Matches the strongest daily seasonal pattern while keeping the parameter search manageable.",
        },
        {
            "model_family": "SARIMA with s=168",
            "use_case": "Weekly seasonal alternative",
            "fit_for_this_project": "Secondary",
            "reason": "Captures weekly repetition directly but is heavier and less practical for iterative notebook fitting.",
        },
    ]
)

display(model_family_comparison)


,model_family,use_case,fit_for_this_project,reason
0,ARIMA,Non-seasonal baseline on differenced data,Weak,Ignores the dominant daily cycle and will like...
1,SARIMA with s=24,Primary hourly forecasting candidate,Strong,Matches the strongest daily seasonal pattern w...
2,SARIMA with s=168,Weekly seasonal alternative,Secondary,Captures weekly repetition directly but is hea...


The earlier notebooks make the family decision fairly clear. A purely non-seasonal ARIMA model is too limited for this dataset, while a weekly seasonal model is possible but much more expensive to search and fit. That makes daily SARIMA the best default family for the next stage.


In [3]:
full_series = pd.read_csv(
    os.path.join(PROCESSED_PATH, "pjme_imputed.csv"),
    parse_dates=["Datetime"],
    index_col="Datetime",
)["PJME_MW"].astype(float)

modeling_window = full_series.iloc[-24 * 180 :]
validation_horizon_hours = 24 * 14

train_series = modeling_window.iloc[:-validation_horizon_hours]
validation_series = modeling_window.iloc[-validation_horizon_hours:]

split_summary = pd.DataFrame(
    [
        {
            "modeling_window_start": modeling_window.index.min(),
            "modeling_window_end": modeling_window.index.max(),
            "train_points": len(train_series),
            "validation_points": len(validation_series),
            "validation_horizon_hours": validation_horizon_hours,
        }
    ]
)

display(split_summary)


,modeling_window_start,modeling_window_end,train_points,validation_points,validation_horizon_hours
0,2018-02-04 01:00:00,2018-08-03,3984,336,336


A rolling subset of recent data keeps later fitting steps practical while still representing the current seasonal pattern. Using a 14-day validation horizon is also reasonable for an hourly load forecasting project because it covers both daily and weekly behavior in the holdout period.


In [4]:
model_identification_summary = pd.DataFrame(
    [
        {
            "selected_model_family": "SARIMA",
            "selected_order_template": "(p,1,q)",
            "selected_seasonal_order_template": "(P,1,Q,24)",
            "primary_seasonal_period": 24,
            "secondary_cycle_hours": 168,
            "recommended_train_window_hours": len(train_series),
            "recommended_validation_window_hours": len(validation_series),
            "reason": "The raw series is non-stationary and strongly seasonal, while the transformed series supports a low-order daily SARIMA search.",
        }
    ]
)

model_identification_summary.to_csv(
    os.path.join(PROCESSED_PATH, "model_identification_summary.csv"), index=False
)

print("Saved model identification summary to ../data/processed/model_identification_summary.csv")
model_identification_summary


Saved model identification summary to ../data/processed/model_identification_summary.csv


,selected_model_family,selected_order_template,selected_seasonal_order_template,primary_seasonal_period,secondary_cycle_hours,recommended_train_window_hours,recommended_validation_window_hours,reason
0,SARIMA,"(p,1,q)","(P,1,Q,24)",24,168,3984,336,The raw series is non-stationary and strongly ...


Final decision: continue with a daily SARIMA specification using `d=1`, `D=1`, and `s=24`, then shortlist a small set of low-order `p`, `q`, `P`, and `Q` values in the next notebook. This keeps the workflow grounded in the data diagnostics instead of relying on a blind parameter sweep.
